# Model 1: RAG (Chuyên xử lý tiếng anh)

File này sẽ tạo ra một Database Vector (`manga_chroma_db`) sử dụng mô hình ngôn ngữ tiếng anh (`all-MiniLM-L6-v2`).

**Mục tiêu:** Giúp Chatbot hiểu được câu hỏi bằng Tiếng Anh với độ chính xác tương đối ổn định. 

In [1]:
!pip install torch --index-url https://download.pytorch.org/whl/cpu

!pip install sentence-transformers chromadb scikit-learn pandas numpy networkx

Looking in indexes: https://download.pytorch.org/whl/cpu


In [2]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
import os
import shutil

# Load data
input_file = 'processed_manga.pkl'
if not os.path.exists(input_file):
    print(f"Loi: Khong tim thay file '{input_file}', can chay file EDA_Analyse.ipynb truoc.")
else:
    df = pd.read_pickle(input_file)

    initial_count = len(df)
    df.drop_duplicates(subset=['mal_id'], keep='first', inplace=True)
    cleaned_count = len(df)
    print(f"Da load file va xu ly trung lap.")    
    
    # Check if search_text is None
    if 'search_text' not in df.columns:
        df['search_text'] = df['title'] + " " + df['genres_str'] + " " + df['synopsis']
    print(f"Load file hoan tat. Bo du lieu gom {len(df)} dong.")

Da load file va xu ly trung lap.
Load file hoan tat. Bo du lieu gom 52247 dong.


In [3]:
# Init database

DB_PATH = "./manga_chroma_db"
if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)

# Create persistent client
chroma_client = chromadb.PersistentClient(path=DB_PATH)

# Create collection of vectors
collection = chroma_client.get_or_create_collection(name="manga_collection")


In [7]:
# Only run this when recreate manga_chroma_db
model = SentenceTransformer('all-MiniLM-L6-v2')

# Data collection
# ChromaDB: IDs (mal_id), Embeddings, Metadatas, Documents (search_text)
documents = df['search_text'].tolist()
ids = [str(x) for x in df['mal_id'].tolist()] # ID: string

# Prepare metadata
metadatas = []
for index, row in df.iterrows():
    meta = {
        "title": str(row['title']),
        "score": float(row['score']),
        "genres": str(row['genres_str']), # Genres: string
        "popularity_score": float(row['popularity_score']), 
        "url": str(row['url'])
    }
    metadatas.append(meta)

# Batch processing
batch_size = 500
total_docs = len(documents)

for i in range(0, total_docs, batch_size):
    end_idx = min(i + batch_size, total_docs)
    batch_docs = documents[i:end_idx]
    batch_ids = ids[i:end_idx]
    batch_metadatas = metadatas[i:end_idx]
    
    # Create vectors
    batch_embeddings = model.encode(batch_docs)
    
    # Storing
    collection.add(
        ids=batch_ids,
        embeddings=batch_embeddings,
        metadatas=batch_metadatas,
        documents=batch_docs
    )

print(f"\nStoring complete.")


Storing complete.


In [8]:
# Test 

DB_PATH = "./manga_chroma_db"
try:
    client = chromadb.PersistentClient(path=DB_PATH)
    collection = client.get_collection("manga_collection")
    print(f"Da ket noi thanh cong voi co so du lieu. Bo du lieu hien co {collection.count()} bo truyen.")
except Exception as e:
    print(f"Loi: Khong ket noi duoc voi co so du lieu. Hay chay lai mo hinh. {e}")

# Reload model 
model = SentenceTransformer('all-MiniLM-L6-v2')

# Chatbot behaviour 
def test_recommendation(user_query, n_results=5):
    print(f"\nUser Query: '{user_query}'")
    print("-" * 50)
    
    # Vectorize questions 
    query_vector = model.encode(user_query).tolist()
    
    # ChromaDB Query
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=n_results,
        where={"popularity_score": {"$gt": 0.5}} 
    )
    
    # Show results
    if not results['ids'][0]:
        print("Khong tim thay ket qua nao phu hop.")
        return

    # Evaluate outputs
    for i in range(len(results['ids'][0])):
        meta = results['metadatas'][0][i]
        dist = results['distances'][0][i] 
        
        print(f"#{i+1} [Similarity: {1-dist:.2f}]")
        print(f"   Title: {meta['title']}")
        print(f"   Score: {meta['score']} | Popularity: {meta['popularity_score']:.2f}")
        print(f"   Genres: {meta['genres']}")
        print(f"   URL: {meta['url']}")
        print("-" * 30)

Da ket noi thanh cong voi co so du lieu. Bo du lieu hien co 52247 bo truyen.


In [9]:
# Test prompt
test_recommendation("I want a manga about ninja and fighting", n_results=3) # Content-based
test_recommendation("Something sad and romantic that makes me cry", n_results=3) # Mood-based
test_recommendation("Pirates looking for treasure", n_results=3) # Synopsis-based

# Semantic Search
query_plot = """
A story about a young boy whose body gained the properties of rubber after eating a devil fruit. 
He gathers a diverse crew of swordsmen, navigators, and cooks to sail the Grand Line. 
They battle the World Government and other pirates to find the ultimate treasure left by the late Pirate King.
"""
test_recommendation(query_plot, n_results=3)

query_theme = """
An epic long-running adventure focusing on dreams, friendship, and freedom on the high seas. 
The protagonist wears a straw hat and refuses to give up on his journey to the final island. 
It features emotional backstories for every crew member and intense battles against a corrupt navy.
"""
test_recommendation(query_theme, n_results=3)

query_tricky = """
A pirate adventure where the main character wants to be the King, 
but he also uses ninja techniques and has a fox monster inside him.
"""
test_recommendation(query_tricky, n_results=5)

query_random_suggestion = """
I enjoy a lot being a child so, so long ago. I want to read some mangas about childhood. 
It should fulfill me with lots of courage and technical gadgets along the way. The story should be about a boy
who wants to be better. As he progress he may have his friends be by his sides. 
"""
test_recommendation(query_random_suggestion, n_results=5)

query_specific_manga = """
I really love Doraemon. Do we have it in the database ?
"""
test_recommendation(query_specific_manga, n_results=5)

query_variants_manga = """
Show me all of your mangas that are variants of the original Detective Conan series.
"""
test_recommendation(query_variants_manga, n_results=10)

query_author_based_manga = """
I want to read more mangas written by Oda
"""
test_recommendation(query_author_based_manga, n_results=10)


User Query: 'I want a manga about ninja and fighting'
--------------------------------------------------
#1 [Similarity: 0.24]
   Title: Under Ninja
   Score: 7.18 | Popularity: 0.54
   Genres: Action, Comedy
   URL: https://myanimelist.net/manga/114939/Under_Ninja
------------------------------
#2 [Similarity: 0.12]
   Title: Road to Ninja: Naruto the Movie
   Score: 7.2 | Popularity: 0.54
   Genres: Action, Adventure, Fantasy
   URL: https://myanimelist.net/manga/40549/Road_to_Ninja__Naruto_the_Movie
------------------------------
#3 [Similarity: 0.12]
   Title: Tenkaichi: Nihon Saikyou Bugeisha Ketteisen
   Score: 7.64 | Popularity: 0.59
   Genres: Action
   URL: https://myanimelist.net/manga/139395/Tenkaichi__Nihon_Saikyou_Bugeisha_Ketteisen
------------------------------

User Query: 'Something sad and romantic that makes me cry'
--------------------------------------------------
#1 [Similarity: -0.18]
   Title: Kekkon shitemo Koishiteru
   Score: 7.4 | Popularity: 0.50
   Genres